# IMQCAM File Collections: Search, Inspect, and Download Data

This notebook shows modelers how to find IMQCAM data files using the IMQCAM-specific `/imqcam/datafiles` endpoint and download only the matching results.

`dataType` is now optional for `/imqcam/datafiles`. This means a modeler can start from the search criteria they know instead of being required to start from a data type. The endpoint itself only exposes items that have IGSN metadata.

The workflow supports:

- **IGSN**
- **Data type**
- **Composition**
- **File extension** (for example `.ang`, `.ctf`, `.csv`)
- combinations of those filters

### IGSN note

Some IMQCAM items store more than one IGSN in the same metadata field, for example:

`JHABOX00001-002, JHABOX00001-001`

To avoid missing those records, this notebook normalizes the stored IGSN values and performs constituent-IGSN matching after the API results are returned.

### Safe defaults

- All search filters may be left blank to retrieve all items exposed by `/imqcam/datafiles`.
- Downloads are **off by default**.
- Downloads use the same filtered results shown in the notebook.


## 1. Connect to the IMQCAM DMS

Run this notebook from the repository so that `imqcam.py` can be imported.


In [ ]:
import json
import re
import sys
from pathlib import Path

import pandas as pd

REPO_ROOT = next(
    p for p in [Path.cwd(), *Path.cwd().parents]
    if (p / "imqcam.py").exists()
)
sys.path.insert(0, str(REPO_ROOT))

from imqcam import get_client

client = get_client()
print(f"Connected to {client.urlBase}")


## 2. Explore available data types

`GET /imqcam/datatype` lists the data types currently available in IMQCAM.


In [ ]:
data_types = client.get("imqcam/datatype")
print(data_types)


## 3. Count files

`GET /imqcam/count` returns file counts. With no parameters, it returns overall counts.


In [ ]:
counts = client.get(
    "imqcam/count",
    parameters={},
)
print(counts)


### Count files for one IGSN

The count endpoint can also be restricted to a specimen IGSN.


In [ ]:
example_igsn = "CMXMAG00001-042"

counts_for_igsn = client.get(
    "imqcam/count",
    parameters={"igsn": example_igsn},
)
print(counts_for_igsn)


## 4. Retrieve data files

`GET /imqcam/datafiles` is the main endpoint used below.

`dataType` is optional. When it is omitted, the endpoint can return all accessible data files, subject to any metadata filters supplied in the request.


In [ ]:
example_items = client.get(
    "imqcam/datafiles",
    parameters={
        "dataType": "ebsd_test",  # optional
        "limit": 5,
    },
)
print(f"Returned {len(example_items)} item(s)")


## 5. Choose search filters

`/imqcam/datafiles` only exposes items that have IGSN metadata. Providing an IGSN as a **search argument** is optional.

Set any filters you know:

- `IGSN`: optional; one IGSN, comma-separated IGSNs, or a Python list
- `DATA_TYPE`: optional; for example `ebsd_raw`
- `COMPOSITION_FILTER`: optional; for example `IN718`
- `FILE_EXTENSION`: optional; for example `.ang`, `.ctf`, or `.csv`

Leave a filter blank to ignore it.

If **all search filters are blank**, the notebook retrieves items exposed by `/imqcam/datafiles`. Items without IGSN metadata are excluded by the endpoint itself.

The search follows this pattern:

`optional API criteria -> /imqcam/datafiles -> local IGSN/composition/extension checks -> filtered results`

`dataType` is sent to `/imqcam/datafiles` only when provided.

Composition is requested from `/imqcam/datafiles` through:

`extraFields=["meta.composition"]`

and then checked locally against each returned item's metadata.

> Note: macOS AppleDouble sidecar files whose names begin with `._` are ignored by the file-extension filter.


In [ ]:
# IGSN is an optional SEARCH filter.
# /imqcam/datafiles itself only exposes items that already have IGSN metadata.
IGSN = ""
DATA_TYPE = ""
COMPOSITION_FILTER = ""
FILE_EXTENSION = ""  # e.g. ".ang", ".ctf", ".csv"

PAGE_SIZE = 100
MAX_RESULTS = None  # Use None to fetch every matching item exposed by the endpoint


## 6. Search helpers

These helpers:

1. normalize one or many requested IGSNs;
2. call `/imqcam/datafiles` with optional `dataType`;
3. request `meta.composition` through `extraFields`;
4. paginate through the endpoint results;
5. perform constituent-IGSN matching locally so comma-separated IGSN metadata is handled correctly;
6. apply composition and file-extension filtering locally.

`meta.composition` cannot be used directly in the endpoint's `filters` parameter, so it is requested through `extraFields` and then filtered locally.


In [ ]:
def split_igsns(value):
    """Normalize one or many IGSNs into a unique, ordered list."""
    if value is None:
        return []

    if isinstance(value, (list, tuple, set)):
        result = []
        for part in value:
            result.extend(split_igsns(part))
        return list(dict.fromkeys(result))

    text = str(value).strip()
    if not text or text.lower() in {"none", "nan"}:
        return []

    return list(
        dict.fromkeys(
            part.strip()
            for part in text.split(",")
            if part.strip()
        )
    )


def item_igsns(item):
    """Return all constituent IGSNs stored on one returned item."""
    meta = item.get("meta", {}) or {}
    return split_igsns(meta.get("igsn"))


def matches_requested_igsn(item, requested_igsns):
    """Return True when any requested IGSN is present on the item."""
    if not requested_igsns:
        return True

    stored = {value.casefold() for value in item_igsns(item)}
    requested = {value.casefold() for value in requested_igsns}

    return bool(stored & requested)


def matches_composition(item, composition):
    """Return True when item metadata matches the requested composition."""
    requested = str(composition or "").strip().casefold()
    if not requested:
        return True

    meta = item.get("meta", {}) or {}
    stored = str(meta.get("composition") or "").strip().casefold()

    return stored == requested


def normalize_extension(value):
    """Normalize 'ang' or '.ang' to '.ang'."""
    text = str(value or "").strip().lower()
    if not text:
        return ""
    return text if text.startswith(".") else f".{text}"


def matches_file_extension(item, extension):
    """Return True when the item name matches the requested file extension."""
    extension = normalize_extension(extension)
    if not extension:
        return True

    name = str(item.get("name") or "").strip().lower()

    # Ignore macOS AppleDouble sidecar files such as "._scan.ctf".
    if name.startswith("._"):
        return False

    return name.endswith(extension)


def fetch_datafiles(
    client,
    igsn="",
    data_type="",
    composition="",
    file_extension="",
    page_size=100,
    max_results=None,
):
    """
    Query /imqcam/datafiles and return items matching the active search filters.

    The endpoint itself only exposes items that have IGSN metadata.
    dataType is optional and is sent to the endpoint when provided.
    meta.composition is requested through extraFields.
    IGSN, composition, and file extension are checked locally.
    """
    requested_igsns = split_igsns(igsn)
    data_type = str(data_type or "").strip()
    composition = str(composition or "").strip()
    file_extension = normalize_extension(file_extension)

    if not requested_igsns and not data_type and not composition and not file_extension:
        print(
            "No search filters supplied: returning items exposed by "
            "/imqcam/datafiles. Items without IGSN metadata are excluded "
            "by the endpoint."
        )

    parameters = {
        "limit": page_size,
        "offset": 0,
        "extraFields": json.dumps(["meta.composition"]),
    }

    if data_type:
        parameters["dataType"] = data_type

    results = []
    offset = 0

    while True:
        parameters["limit"] = page_size
        parameters["offset"] = offset

        batch = client.get(
            "imqcam/datafiles",
            parameters=parameters,
        )

        if not batch:
            break

        for item in batch:
            if (
                matches_requested_igsn(item, requested_igsns)
                and matches_composition(item, composition)
                and matches_file_extension(item, file_extension)
            ):
                results.append(item)

                if max_results is not None and len(results) >= max_results:
                    return results[:max_results]

        if len(batch) < page_size:
            break

        offset += len(batch)

    return results


## 7. Run the search

`/imqcam/datafiles` performs the API query first. The endpoint only exposes items that have IGSN metadata.

The notebook also requests `meta.composition` using `extraFields`.

If an IGSN, composition, or file extension is supplied, the notebook applies those checks to the returned items. Comma-separated IGSN metadata is handled as constituent values.

The final `items` list is the same list used later for display and download.


In [ ]:
items = fetch_datafiles(
    client=client,
    igsn=IGSN,
    data_type=DATA_TYPE,
    composition=COMPOSITION_FILTER,
    file_extension=FILE_EXTENSION,
    page_size=PAGE_SIZE,
    max_results=MAX_RESULTS,
)

selected_filters = []

if split_igsns(IGSN):
    selected_filters.append(f"IGSN={', '.join(split_igsns(IGSN))}")

if str(DATA_TYPE or "").strip():
    selected_filters.append(f"data type={DATA_TYPE}")

if str(COMPOSITION_FILTER or "").strip():
    selected_filters.append(f"composition={COMPOSITION_FILTER}")

if normalize_extension(FILE_EXTENSION):
    selected_filters.append(
        f"file extension={normalize_extension(FILE_EXTENSION)}"
    )

if selected_filters:
    print(
        f"Matched {len(items):,} item(s) for selected filters: "
        + ", ".join(selected_filters)
    )
else:
    print(
        f"Matched {len(items):,} item(s) with no search filters "
        "(items exposed by /imqcam/datafiles)."
    )


## 8. Inspect matching metadata

Review the matches before downloading. The table below shows the key metadata fields for each returned item.


In [ ]:
rows = []

for item in items:
    meta = item.get("meta", {}) or {}

    rows.append(
        {
            "name": item.get("name"),
            "item_id": item.get("_id"),
            "igsn": meta.get("igsn"),
            "composition": meta.get("composition"),
            "data_type": meta.get("data_type"),
            "extension": Path(str(item.get("name") or "")).suffix.lower(),
        }
    )

results_df = pd.DataFrame(
    rows,
    columns=["name", "item_id", "igsn", "composition", "data_type", "extension"],
)

display(results_df.head(20))


## 9. Download the matched items

Downloads are disabled by default.

Review `results_df` first. When the results are correct, set:

```python
DOWNLOAD_FILES = True
```

Only the `items` returned by the active metadata query are downloaded.

`DOWNLOAD_LIMIT` is set to `5` by default as an additional safeguard. Set it to `None` only when you intend to download every matched item.


In [ ]:
DOWNLOAD_FILES = False
DOWNLOAD_LIMIT = 5  # Set to None to download every matched item


def safe_folder_name(value, fallback):
    """Create a simple filesystem-safe folder component."""
    text = str(value or "").strip() or fallback
    return re.sub(r"[^A-Za-z0-9._-]+", "_", text)


if DOWNLOAD_FILES:
    items_to_download = items

    if DOWNLOAD_LIMIT is not None:
        items_to_download = items_to_download[:DOWNLOAD_LIMIT]

    requested_igsns = split_igsns(IGSN)

    if not requested_igsns:
        igsn_folder = "all_igsns"
    elif len(requested_igsns) == 1:
        igsn_folder = requested_igsns[0]
    else:
        igsn_folder = "multiple_igsns"

    download_folder = (
        REPO_ROOT
        / "downloads"
        / safe_folder_name(DATA_TYPE, "all_data_types")
        / safe_folder_name(igsn_folder, "all_igsns")
        / safe_folder_name(COMPOSITION_FILTER, "all_compositions")
        / safe_folder_name(FILE_EXTENSION, "all_extensions")
    )
    download_folder.mkdir(parents=True, exist_ok=True)

    print(f"Downloading {len(items_to_download):,} matched item(s)...")
    print(f"Destination: {download_folder}")

    for number, item in enumerate(items_to_download, start=1):
        item_id = item.get("_id")
        item_name = item.get("name", item_id)

        if not item_id:
            print(f"Skipping item without an _id: {item_name}")
            continue

        client.downloadItem(
            item_id,
            dest=str(download_folder),
        )

        print(f"[{number}/{len(items_to_download)}] {item_name}")

    print("Download complete.")

else:
    print(
        "Downloads are OFF. Review the matched items first, then set "
        "DOWNLOAD_FILES = True when ready."
    )


## Summary

For most modeler searches, use Sections 5–9:

1. choose any search filters you know;
2. query `/imqcam/datafiles` with optional `dataType`;
3. request `meta.composition` through `extraFields`;
4. apply constituent-IGSN matching locally when an IGSN is supplied;
5. apply composition and file-extension checks locally;
6. inspect the returned metadata;
7. enable downloads only after confirming the matches.

`dataType` is optional. Providing an IGSN as a search argument is also optional, but `/imqcam/datafiles` itself only exposes items that already have IGSN metadata.

Composition is requested with `extraFields=["meta.composition"]` and filtered locally.

The same final `items` list is used for both display and download.
